<a href="https://colab.research.google.com/github/ShinAsakawa/ShinAsakawa.github.io/blob/master/2026notebooks/2026_0413EM_consistency_NTTpsylex71.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NTT日本語語彙特性データ 単語頻度データ `psylex71.txt` のアップロード
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving psylex71.txt to psylex71.txt
User uploaded file "psylex71.txt" with length 28676843 bytes


In [6]:
# 日本語文字コードの正規化 normalize について
try:
    import jaconv
except ImportError:
    !pip install jaconv
    import jaconv

_str = 'ﾃｨﾛ･フィナ〜レ'
print(_str, '->', jaconv.normalize(_str))

ﾃｨﾛ･フィナ〜レ -> ティロ・フィナーレ


In [19]:
import re

import jaconv
# import os
# HOME = os.environ['HOME']
# psylex71_path = os.path.join(HOME, 'study/2017_2009AmanoKondo_NTTKanjiData/psylex71utf8.txt')
# HOME = os.environ['HOME']
psylex71_path = 'psylex71.txt'

def load_psylex71_data(psylex71_path:str):
    kata_chars = 'ァアィイゥウェエォオカガキギクグケゲコゴサザシジスズセゼソゾタダチヂッツヅテデトドナニヌネノハバパヒビピフブプヘベペホボポマミムメモャヤュユョヨラリルレロヮワヰヱヲンヴヵ゛゛゜ー—'
    kata_firts = 'アイウエオカガキギクグケゲコゴサザシジスズセゼソゾタダチヂッツヅテデトドナニヌネノハバパヒビピフブプヘベペホボポマミムメモヤユヨラリルレロワヰヱヲンヴヵ'
    KANJI_RE = re.compile(r"^[一-龯]{2}$")  # 2 文字漢字限定
    KANJI_RE = re.compile(r"^[一-龯]{2,4}$")  # 2 文字から 4 文字漢字限定
    JAPANESE_CHAR_PATTERN = re.compile(r"^[ぁ-ゖァ-ヺ一-鿿㐀-䶵々]{min_len,max_len}$")
    JAPANESE_CHAR_PATTERN = re.compile(r"^[ぁ-ゖァ-ヺ一-鿿㐀-䶵々]{2,4}$")
    #JAPANESE_CHAR_PATTERN = re.compile(r"^[ぁ-ゖァ-ヺ一-鿿㐀-䶵々]{min_len,max_len}$")

    data = []
    with open(psylex71_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(' ')  # 一行を空白で分割

            word = jaconv.normalize(parts[2])  # 表記
            yomi = jaconv.normalize(parts[3])  # 読み
            pos  = parts[4]  # 品詞
            try:
                freq = int(parts[5])       # 頻度全体
            except ValueError:
                continue

            # Filter: 上で定義した漢字単語だけを取り出す
            if not JAPANESE_CHAR_PATTERN.match(word):
                continue

            is_valid_yomi = False
            if yomi[0] in kata_firts:
                is_valid_yomi = True
                for y in yomi[1:]:
                    if not y in kata_chars:
                        is_valid_yomi = False
            if is_valid_yomi:
                data.append((word, yomi,freq))

    print(f"Loaded {len(data)} kanji words from {psylex71_path}")
    #print(f"Loaded {len(data)} 2-kanji words from {psylex71_path}")
    return data

psylex_data = load_psylex71_data(psylex71_path)
print(f'len(psylex_data):{len(psylex_data)}')


# CDP+ 用（頻度は別で保持）
DATA = [(word, yomi) for (word, yomi, freq) in psylex_data]

# frequency dictionary
word_freq = {word: freq for (word, yomi, freq) in psylex_data}

# KANJI_CONS = build_kanji_consistency(DATA)

Loaded 211745 kanji words from psylex71.txt
len(psylex_data):211745


In [ ]:
"""
EM-based kanji–reading alignment and nonword reading prediction (v4).

Two functional layers:
  1. EM alignment: learns P(reading | kanji) from a corpus of N-kanji words
  2. Nonword prediction: applies learned P(reading | kanji) as a sub-lexical
     route approximation (CDP-style GPC layer)

Diagnostic scores:
  - confidence:       alignment quality (segmentation posterior entropy)
  - align_loglik:     alignment plausibility (per-kanji mean log-likelihood)
  - consistency:      max P(r|K) — dominance of the most frequent reading
  - reading_entropy:  H(P(r|K)) — uncertainty over readings
"""

import math
from collections import defaultdict, Counter
from itertools import combinations, product
from typing import List, Tuple, Dict, Optional, NamedTuple


# ═════════════════════════════════════════════
# Mora segmentation
# ═════════════════════════════════════════════
SMALL_KANA = set(list("ゃゅょぁぃぅぇぉャュョァィゥェォ"))
LONG_VOWEL = "ー"
SOKUON = {"っ", "ッ"}
HATSUON = {"ん", "ン"}


def kana_to_mora(reading: str) -> List[str]:
    morae: List[str] = []
    i = 0
    while i < len(reading):
        ch = reading[i]
        if ch in SOKUON or ch in HATSUON:
            morae.append(ch)
            i += 1
        elif i + 1 < len(reading) and reading[i + 1] in SMALL_KANA:
            morae.append(ch + reading[i + 1])
            i += 2
        elif ch == LONG_VOWEL:
            morae.append(ch)
            i += 1
        else:
            morae.append(ch)
            i += 1
    return morae


# ═════════════════════════════════════════════
# Utilities
# ═════════════════════════════════════════════
def _log_sum_exp(vals: List[float]) -> float:
    if not vals:
        return -float('inf')
    mx = max(vals)
    if mx == -float('inf'):
        return -float('inf')
    return mx + math.log(sum(math.exp(v - mx) for v in vals))


def _entropy(probs: List[float]) -> float:
    return -sum(p * math.log(p + 1e-30) for p in probs if p > 0)


def _confidence(probs: List[float]) -> float:
    n = len(probs)
    if n <= 1:
        return 1.0
    H = _entropy(probs)
    H_max = math.log(n)
    if H_max < 1e-30:
        return 1.0
    return max(0.0, 1.0 - H / H_max)


def _enumerate_segmentations(M: int, N: int) -> List[Tuple[int, ...]]:
    if N > M or N < 1:
        return []
    if N == 1:
        return [()]
    return [sp for sp in combinations(range(1, M), N - 1)]


def _splits_to_segments(M: int, splits: Tuple[int, ...]) -> List[Tuple[int, int]]:
    boundaries = [0] + list(splits) + [M]
    return [(boundaries[i], boundaries[i + 1]) for i in range(len(boundaries) - 1)]


# ═════════════════════════════════════════════
# Layer 1: EM alignment
# ═════════════════════════════════════════════
def _build_entries(data: List[Tuple[str, str]]):
    entries = []
    for word, yomi in data:
        kanji_chars = list(word)
        N = len(kanji_chars)
        morae = kana_to_mora(yomi)
        M = len(morae)
        if M < N:
            continue
        segs = _enumerate_segmentations(M, N)
        if not segs:
            continue
        entries.append((word, yomi, kanji_chars, morae, segs))
    return entries


def em_align(
    data: List[Tuple[str, str]],
    n_iter: int = 20,
    smoothing: float = 1e-8,
    freq_weights: Optional[Dict[str, int]] = None,
    use_confidence_weighting: bool = True,
) -> Tuple[
    Dict[str, Counter],
    Dict[Tuple[str, str], Tuple[Tuple[str, ...], ...]],
    Dict[Tuple[str, str], float],
    Dict[Tuple[str, str], float],
]:
    """
    Returns:
        kanji_dist, alignments, confidences, align_logliks
    """
    entries = _build_entries(data)
    if not entries:
        return {}, {}, {}, {}

    def _log_emit(kanji, reading):
        total = sum(kanji_dist[kanji].values()) + smoothing
        count = kanji_dist[kanji].get(reading, 0) + smoothing
        return math.log(count / total)

    def _seg_log_prob(kanji_chars, morae, splits):
        segments = _splits_to_segments(len(morae), splits)
        return sum(
            _log_emit(kanji_chars[i], tuple(morae[a:b]))
            for i, (a, b) in enumerate(segments)
        )

    # Initialize
    kanji_dist: Dict[str, Counter] = defaultdict(Counter)
    for word, yomi, kanji_chars, morae, segs in entries:
        w = (freq_weights.get(word, 1) if freq_weights else 1)
        n_segs = len(segs)
        uniform_w = w / n_segs
        for splits in segs:
            segments = _splits_to_segments(len(morae), splits)
            for i, (a, b) in enumerate(segments):
                kanji_dist[kanji_chars[i]][tuple(morae[a:b])] += uniform_w

    # EM
    for iteration in range(n_iter):
        new_dist: Dict[str, Counter] = defaultdict(Counter)
        for word, yomi, kanji_chars, morae, segs in entries:
            w = (freq_weights.get(word, 1) if freq_weights else 1)
            log_probs = [_seg_log_prob(kanji_chars, morae, sp) for sp in segs]
            lse = _log_sum_exp(log_probs)
            posteriors = [math.exp(lp - lse) for lp in log_probs]
            conf = _confidence(posteriors) if use_confidence_weighting else 1.0
            for sp, post in zip(segs, posteriors):
                segments = _splits_to_segments(len(morae), sp)
                for i, (a, b) in enumerate(segments):
                    new_dist[kanji_chars[i]][tuple(morae[a:b])] += post * w * conf
        kanji_dist = new_dist

    # MAP alignment
    alignments = {}
    confidences = {}
    align_logliks = {}
    for word, yomi, kanji_chars, morae, segs in entries:
        log_probs = [_seg_log_prob(kanji_chars, morae, sp) for sp in segs]
        lse = _log_sum_exp(log_probs)
        posteriors = [math.exp(lp - lse) for lp in log_probs]
        best_idx = max(range(len(segs)), key=lambda i: log_probs[i])
        best_segments = _splits_to_segments(len(morae), segs[best_idx])
        key = (word, yomi)
        alignments[key] = tuple(tuple(morae[a:b]) for a, b in best_segments)
        confidences[key] = _confidence(posteriors)
        align_logliks[key] = log_probs[best_idx] / len(kanji_chars)

    return dict(kanji_dist), alignments, confidences, align_logliks


# ═════════════════════════════════════════════
# Kanji-level statistics from learned dist
# ═════════════════════════════════════════════
class KanjiStats(NamedTuple):
    """Per-kanji reading statistics."""
    consistency: float    # max P(r|K) — reading dominance
    entropy: float        # H(P(r|K)) in nats — reading uncertainty
    n_readings: int       # number of distinct readings with non-negligible mass
    top_reading: Tuple[str, ...]   # most probable reading (mora tuple)
    top_prob: float       # probability of top reading


def kanji_stats(kanji: str, kanji_dist: Dict[str, Counter],
                min_prob: float = 0.01) -> Optional[KanjiStats]:
    """Compute reading statistics for a single kanji."""
    if kanji not in kanji_dist:
        return None
    cnt = kanji_dist[kanji]
    total = sum(cnt.values())
    if total <= 0:
        return None

    probs = {r: c / total for r, c in cnt.items()}
    top_reading = max(probs, key=probs.get)
    top_prob = probs[top_reading]
    prob_list = list(probs.values())

    return KanjiStats(
        consistency=top_prob,
        entropy=_entropy(prob_list),
        n_readings=sum(1 for p in prob_list if p >= min_prob),
        top_reading=top_reading,
        top_prob=top_prob,
    )


def build_consistency_from_dist(
    kanji_dist: Dict[str, Counter],
) -> Dict[str, float]:
    consistency = {}
    for kanji, cnt in kanji_dist.items():
        total = sum(cnt.values())
        consistency[kanji] = (max(cnt.values()) / total) if total > 0 else 0.0
    return consistency


# ═════════════════════════════════════════════
# Layer 2: Nonword reading prediction
#   (Sub-lexical route approximation)
# ═════════════════════════════════════════════
class NonwordReading(NamedTuple):
    """Prediction result for a single nonword."""
    reading: str                      # concatenated mora string (argmax)
    per_kanji: Tuple[Tuple[str, ...], ...]  # reading tuple per kanji
    joint_prob: float                 # product of per-kanji max probs
    joint_logprob: float              # sum of per-kanji log probs
    per_kanji_consistency: Tuple[float, ...]  # consistency per kanji
    per_kanji_entropy: Tuple[float, ...]      # entropy per kanji
    word_consistency: float           # mean consistency across kanji
    word_entropy: float               # mean entropy across kanji


def predict_nonword(
    nonword: str,
    kanji_dist: Dict[str, Counter],
    smoothing: float = 1e-8,
) -> NonwordReading:
    """
    Predict the most likely reading of a nonword (or any kanji string)
    using per-kanji reading distributions learned by EM.

    This approximates the CDP sub-lexical (GPC) route:
      - Each kanji independently activates its reading distribution
      - The argmax reading per kanji is selected
      - No lexical feedback, no inter-kanji context
    """
    per_kanji_readings = []
    per_kanji_probs = []
    per_kanji_cons = []
    per_kanji_ent = []

    for k in nonword:
        cnt = kanji_dist.get(k, Counter())
        total = sum(cnt.values())

        if total <= 0:
            # Unknown kanji: no data
            per_kanji_readings.append(("？",))
            per_kanji_probs.append(smoothing)
            per_kanji_cons.append(0.0)
            per_kanji_ent.append(0.0)
            continue

        probs = {r: c / total for r, c in cnt.items()}
        top_r = max(probs, key=probs.get)
        top_p = probs[top_r]
        prob_list = list(probs.values())

        per_kanji_readings.append(top_r)
        per_kanji_probs.append(top_p)
        per_kanji_cons.append(top_p)  # consistency = max prob
        per_kanji_ent.append(_entropy(prob_list))

    # Joint probability
    joint_logp = sum(math.log(p + 1e-30) for p in per_kanji_probs)
    joint_p = math.exp(joint_logp)

    # Concatenate reading
    reading_str = "".join("".join(r) for r in per_kanji_readings)

    n = len(nonword)
    return NonwordReading(
        reading=reading_str,
        per_kanji=tuple(per_kanji_readings),
        joint_prob=joint_p,
        joint_logprob=joint_logp,
        per_kanji_consistency=tuple(per_kanji_cons),
        per_kanji_entropy=tuple(per_kanji_ent),
        word_consistency=sum(per_kanji_cons) / n if n > 0 else 0.0,
        word_entropy=sum(per_kanji_ent) / n if n > 0 else 0.0,
    )


def predict_nonword_topn(
    nonword: str,
    kanji_dist: Dict[str, Counter],
    top_n: int = 5,
) -> List[Tuple[str, Tuple[Tuple[str, ...], ...], float]]:
    """
    Return top-N reading candidates with joint probabilities.

    Generates candidates from the Cartesian product of per-kanji top readings.
    Useful for modeling reading competition (activation overlap in GPC output).

    Returns:
        list of (reading_str, per_kanji_readings, joint_prob)
        sorted by joint_prob descending
    """
    per_kanji_tops = []
    for k in nonword:
        cnt = kanji_dist.get(k, Counter())
        total = sum(cnt.values())
        if total <= 0:
            per_kanji_tops.append([(("？",), 1.0)])
            continue
        ranked = [(r, c / total) for r, c in cnt.most_common(top_n)]
        per_kanji_tops.append(ranked)

    candidates = []
    for combo in product(*per_kanji_tops):
        readings, probs = zip(*combo)
        joint_p = 1.0
        for p in probs:
            joint_p *= p
        reading_str = "".join("".join(r) for r in readings)
        candidates.append((reading_str, readings, joint_p))

    candidates.sort(key=lambda x: -x[2])
    return candidates[:top_n]


# ═════════════════════════════════════════════
# Inspection
# ═════════════════════════════════════════════
def show_kanji_readings(kanji: str, kanji_dist: Dict[str, Counter], top_n: int = 10):
    if kanji not in kanji_dist:
        print(f"'{kanji}' not found.")
        return
    cnt = kanji_dist[kanji]
    total = sum(cnt.values())
    ranked = cnt.most_common(top_n)
    print(f"漢字 '{kanji}' — 読み分布 (total={total:.1f}):")
    for mora_tuple, count in ranked:
        reading_str = "".join(mora_tuple)
        pct = 100 * count / total
        print(f"  {reading_str:8s}  {count:8.1f}  ({pct:5.1f}%)")


def show_alignment(word, yomi, alignments, confidences=None, align_logliks=None):
    key = (word, yomi)
    if key not in alignments:
        print(f"({word}, {yomi}) not found.")
        return
    segs = alignments[key]
    parts = [f"{c}→{''.join(r)}" for c, r in zip(word, segs)]
    extras = []
    if confidences and key in confidences:
        extras.append(f"conf={confidences[key]:.3f}")
    if align_logliks and key in align_logliks:
        extras.append(f"loglik={align_logliks[key]:.3f}")
    extra_str = "  " + "  ".join(extras) if extras else ""
    print(f"{'  '.join(parts)}  ({word}/{yomi}){extra_str}")


def show_nonword_prediction(nonword: str, kanji_dist: Dict[str, Counter]):
    """Pretty-print nonword prediction."""
    result = predict_nonword(nonword, kanji_dist)
    print(f"非単語: {nonword}")
    print(f"  最尤読み: {result.reading}")
    print(f"  漢字別:")
    for i, k in enumerate(nonword):
        r = "".join(result.per_kanji[i])
        c = result.per_kanji_consistency[i]
        h = result.per_kanji_entropy[i]
        print(f"    {k} → {r:6s}  consistency={c:.3f}  entropy={h:.3f}")
    print(f"  結合確率:     {result.joint_prob:.6f}")
    print(f"  結合対数尤度: {result.joint_logprob:.3f}")
    print(f"  語一貫性:     {result.word_consistency:.3f}")
    print(f"  語エントロピー: {result.word_entropy:.3f}")

    # Top-N candidates
    topn = predict_nonword_topn(nonword, kanji_dist, top_n=5)
    print(f"  読み候補:")
    for rank, (reading, per_k, jp) in enumerate(topn, 1):
        per_k_str = " + ".join("".join(r) for r in per_k)
        print(f"    {rank}. {reading:10s} ({per_k_str})  P={jp:.6f}")
    print()


# ═════════════════════════════════════════════
# Demo
# ═════════════════════════════════════════════
if __name__ == "__main__":
    # ── Corpus data (real words for EM training) ──
    corpus = [
        ("先生", "センセイ"), ("学生", "ガクセイ"), ("学校", "ガッコウ"),
        ("学会", "ガッカイ"), ("先日", "センジツ"), ("先端", "センタン"),
        ("校長", "コウチョウ"), ("校舎", "コウシャ"), ("生活", "セイカツ"),
        ("生命", "セイメイ"), ("発表", "ハッピョウ"), ("発見", "ハッケン"),
        ("表面", "ヒョウメン"), ("表現", "ヒョウゲン"), ("熱湯", "ネットウ"),
        ("熱心", "ネッシン"), ("今朝", "ケサ"), ("今日", "キョウ"),
        ("今回", "コンカイ"), ("追求", "ツイキュウ"), ("追加", "ツイカ"),
        ("求人", "キュウジン"), ("図書館", "トショカン"),
        ("博物館", "ハクブツカン"), ("美術館", "ビジュツカン"),
        ("不思議", "フシギ"), ("遊園地", "ユウエンチ"),
        ("活動", "カツドウ"), ("活発", "カッパツ"), ("活用", "カツヨウ"),
        ("表示", "ヒョウジ"), ("発熱", "ハツネツ"), ("発生", "ハッセイ"),
        ("発電", "ハツデン"), ("生産", "セイサン"), ("生徒", "セイト"),
        ("会長", "カイチョウ"), ("会議", "カイギ"), ("会社", "カイシャ"),
        ("校内", "コウナイ"), ("校門", "コウモン"), ("校庭", "コウテイ"),
        ("熱帯", "ネッタイ"), ("熱量", "ネツリョウ"), ("熱中", "ネッチュウ"),
        ("求職", "キュウショク"), ("求道", "キュウドウ"),
        ("追跡", "ツイセキ"), ("追放", "ツイホウ"),
        ("端末", "タンマツ"), ("端的", "タンテキ"),
    ]

    print("=" * 60)
    print("EM alignment v4 — nonword prediction demo")
    print("=" * 60)

    kanji_dist, alignments, confidences, align_logliks = em_align(
        corpus, n_iter=30, use_confidence_weighting=True
    )

    # ── Show some learned distributions ──
    print("\n── Learned P(reading|kanji) ──")
    for k in ["生", "発", "熱", "活", "会"]:
        show_kanji_readings(k, kanji_dist, top_n=5)
        st = kanji_stats(k, kanji_dist)
        if st:
            print(f"  consistency={st.consistency:.3f}  "
                  f"entropy={st.entropy:.3f}  "
                  f"n_readings={st.n_readings}")
        print()

    # ── Nonword predictions (sub-lexical route) ──
    print("=" * 60)
    print("Nonword predictions (sub-lexical route)")
    print("=" * 60)
    print()

    nonwords = ["熱校", "発求", "活端", "生追", "会熱"]
    for nw in nonwords:
        show_nonword_prediction(nw, kanji_dist)


In [32]:
%%time
DATA
kanji_dist, alignments, confidences, align_logliks = em_align(
        DATA[:], n_iter=30, use_confidence_weighting=True
    )

CPU times: user 24min 59s, sys: 1.59 s, total: 25min
Wall time: 25min 30s


In [33]:
KABC_list = ['先生','糸','石','早い','顔','帰る','歌','遠い','橋','通す','発表','遊園地','不思議','季節','転ぶ', '土手','話題','殺虫ざい','整える','熱湯','今朝','歯科医院', '都合','案の定','追求','財布','快い']

for wrd in KABC_list:
    show_nonword_prediction(wrd, kanji_dist)

非単語: 先生
  最尤読み: サキセイ
  漢字別:
    先 → サキ      consistency=0.536  entropy=0.871
    生 → セイ      consistency=0.376  entropy=2.378
  結合確率:     0.201796
  結合対数尤度: -1.600
  語一貫性:     0.456
  語エントロピー: 1.624
  読み候補:
    1. サキセイ       (サキ + セイ)  P=0.201796
    2. センセイ       (セン + セイ)  P=0.161623
    3. サキオ        (サキ + オ)  P=0.070577
    4. サキウ        (サキ + ウ)  P=0.056977
    5. センオ        (セン + オ)  P=0.056527

非単語: 糸
  最尤読み: イト
  漢字別:
    糸 → イト      consistency=0.770  entropy=0.593
  結合確率:     0.770115
  結合対数尤度: -0.261
  語一貫性:     0.770
  語エントロピー: 0.593
  読み候補:
    1. イト         (イト)  P=0.770115
    2. シ          (シ)  P=0.218391
    3. ス          (ス)  P=0.005747
    4. リ          (リ)  P=0.005747
    5. イイト        (イイト)  P=0.000000

非単語: 石
  最尤読み: イシ
  漢字別:
    石 → イシ      consistency=0.616  entropy=1.299
  結合確率:     0.616333
  結合対数尤度: -0.484
  語一貫性:     0.616
  語エントロピー: 1.299
  読み候補:
    1. イシ         (イシ)  P=0.616333
    2. セキ         (セキ)  P=0.222205
    3. セッ         (セッ)  P=0.044689
    4.